# MQRBN8 Processing

- Whole genome sequence data; Illumina only.
- Shipped and submitted to Plasmidsaurus 12/04/2025.
- Data available on:
- Started procecessing on:

## Steps
1. Download, data organization, file renaming
2. Creating seqsamples and cross-checking in LIMS.
3. QA/QC
4. Breseq (refseq: ACN2586) and Breseq analysis

## 0. Set-up

In [ ]:
## Make sure running in aisynbio_env

In [ ]:
# Reload magic command to ensure that changes made to my imported modules are being picked up by the notebook continuously

%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import os

# Add project root to path for access to workflows and tasks
notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir)

if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [ ]:
import os

# Must add environment's bin to the PATH inside the notebook
env_bin = os.path.join(os.path.dirname(sys.executable), "")
os.environ["PATH"] = env_bin + ":" + os.environ["PATH"]

## 1. Download, file organization, renaming
- Create seqorder name and makedir into reception folder (/synbio/ai_synbio_data/experimental_data/downloads - should this be temporary?). Also make homedir to copy and view analysis reports.
- Download data into folder.
- Spot-check if duplicate read ID problem is fixed.
- Create new seqorder folder in experimental_data/sequencing_data/ and libraries.
- Copy all nanopore fastqs into long lib, renaming in the process.
- Copy all illumina fastqs into short lib, renaming in the process.
Creating sample seqs and cross-checking with LIMS


In [ ]:
# Create seqorder name, make reception folder and seqorder analysis folder in nspahr homedir

from aisynbiopipeline.workflows.plasmidsaurus import create_seqorder_name
from aisynbiopipeline.workflows.seq_folder_utils import list_seqorders, SeqOrder, Library, SeqSample
import os

item_code = 'MQRBN8'
seqorder_name = create_seqorder_name(item_code)

reception_dir = '/storage/synbio/ai_synbio_data/experimental_data/downloads/' + seqorder_name
os.makedirs(reception_dir)
print(reception_dir)

home_dir = '/storage/nspahr/lib_analysis/' + seqorder_name
breseq_pop_dir = home_dir + '/breseq_analysis_populations'
breseq_col_dir = home_dir + '/breseq_analysis_colonies'
os.makedirs(home_dir) 
os.makedirs(breseq_pop_dir)
os.makedirs(breseq_col_dir)


# Summary and report files for review will have this folder structure. Later, check size of folder, possibly zip before downloading to laptop, then uploading to LIMS

# home_dir/
# ├── read_QC_report.html
# ├── breseq_analysis_populations/
# │   ├── run_summary.csv
# │   ├── all_muations.html
# │   ├── all_mutations.csv (can delete this one after download)
# │   ├── all_mutations_reformatted.csv
# │   ├── summary_all_samples.txt (conmut in all, conmut in some)
# │   └── example_sample_X/
# │       ├── all_mutations_example_sample_X.csv
# │       ├── summary_example_sample_X.txt (conmut in all, conmut in some, increasing pol)
# │       └── polymorphisms_increasing_over_transfers.pdf
# └── breseq_analysis_colonies (IN PLANNING)

In [ ]:
# Download data into folder

from aisynbiopipeline.workflows.plasmidsaurus import get_access_token, download_results, get_credentials

CLIENT_ID = get_credentials("PLASMIDSAURUS_CLIENT_ID")
CLIENT_SECRET = get_credentials("PLASMIDSAURUS_CLIENT_SECRET")
access_token = get_access_token(CLIENT_ID, CLIENT_SECRET)
download_results(item_code, access_token, reception_dir)

In [ ]:
os.listdir(reception_dir)

In [ ]:
# Spot-check if duplicate read ID problem is fixed

plasmidsaurus_read_folder_name = item_code + '_reads'
os.listdir(os.path.join(reception_dir, plasmidsaurus_read_folder_name))

In [ ]:
!gunzip -c {os.path.join(reception_dir, plasmidsaurus_read_folder_name, 'P4CYGL_3_ANLstock.ACN2586.colony3_illumina_R1.fastq.gz')} | head

In [ ]:
!gunzip -c {os.path.join(reception_dir, plasmidsaurus_read_folder_name, 'P4CYGL_3_ANLstock.ACN2586.colony3_illumina_R1.fastq.gz')} | grep "@LH00941:53:22CCH7LT1:1:1101:1351:1128 1:N:0:AGTCAGAC+TGTCGCTG"

[Sentence about read duplication result.]

In [ ]:
# Create new seqorder folder in experimental_data/sequencing_data/ and libraries

seqorder = SeqOrder(seqorder_name, create=True)
short = Library(seqorder, 'Illumina', create=True)
# long = Library(seqorder, 'Nanopore', create=True)

In [ ]:
# Identify Illumina/ Nanopore reads and copy into respective library folder

from pathlib import Path
import shutil

reads_path = Path(os.path.join(reception_dir, f'{item_code}_reads'))
i_pattern = "*illumina*.fastq.gz"
n_pattern = "*nanopore*.fastq.gz"
i_files = list(reads_path.glob(i_pattern))
n_files = list(reads_path.glob(n_pattern))

def rename_plasmidsaurus_read_file(file_name):
    aisynbio_filename = ('_').join(file_name.split('_')[2:])
    return aisynbio_filename

for file in i_files:
    plasmidsaurus_basename = os.path.basename(file)
    aisynbio_basename = rename_plasmidsaurus_read_file(plasmidsaurus_basename)
    shutil.copy2(reads_path/plasmidsaurus_basename, short.path/'received'/aisynbio_basename)

for file in n_files:
    plasmidsaurus_basename = os.path.basename(file)
    aisynbio_basename = rename_plasmidsaurus_read_file(plasmidsaurus_basename)
    shutil.copy2(reads_path/plasmidsaurus_basename, long.path/'received'/aisynbio_basename)

In [ ]:
aisynbio_basename

## 2. Cross-checking this Plasmidsaurus order seqsamples in LIMS and creating SeqSamples.

In [ ]:
# Is this really necessary????????????????????????????????????????????????????????????????

# Import LIMS utilities (use util_simple.py for standalone LIMS API usage)
%run util_simple.py

In [ ]:
query_lims(
    'Measurements',
    filters={'Experiment': 'TFMN1', 'Type': 'Short_DNA_reads'}
)

In [ ]:
query_lims(
    'Measurements',
    filters={'Experiment': 'TFMN1', 'Type': 'Short_DNA_reads'}
)

In [ ]:
# Cross-checking seq sample measurement names

short_manifest = short.create_manifest('received')
print(f"Are all short Plasmidsaurus seqsamples from order {item_code} in the LIMS?")
print(all([x in thisExpLIMSseqsamples_short for x in short_manifest['sample_name'].to_list()]))
print(f"Are all short LIMS seqsamples with selected filters in Plasmidsaurus order {item_code}?")
print(all([x in short_manifest['sample_name'].to_list() for x in thisExpLIMSseqsamples_short]))
# long_manifest = long.create_manifest('received')
# print(f"Are all Plasmidsaurus long seqsamples from order {item_code} in the LIMS?")
# print(all([x in thisExpLIMSseqsamples_long for x in long_manifest['sample_name'].to_list()]))
# print(f"Are all LIMS long seqsamples with selected filters in Plasmidsaurus order {item_code}?")
# print(all([x in long_manifest['sample_name'].to_list() for x in thisExpLIMSseqsamples_long]))

In [ ]:
# Creating batch (list) of seqsamples for this seqorder

seqsamples = [SeqSample(short, row['sample_name']) for _, row in short_manifest.iterrows()]

## 3. Short reads: QA/QC

- fastp (Celery): Must start running workers first
- MultiQC

In [ ]:
## fastp worker already running:

(aisynbio_env) nspahr@seed:~/code/AISynbioPipeline$ python -m aisynbiopipeline.tasks.fastp_task 1
======================================================================
Starting fastp Celery Worker
======================================================================

Monitor at: http://poplar.cels.anl.gov:5555
======================================================================

 
 -------------- fastp_1@seed.jupyter v5.6.0 (recovery)
--- ***** ----- 
-- ******* ---- Linux-6.11.0-21-generic-x86_64-with-glibc2.39 2025-12-15 21:34:57
- *** --- * --- 
- ** ---------- [config]
- ** ---------- .> app:         fastp:0x76719a4725a0
- ** ---------- .> transport:   redis://bioseed_redis:6379/10
- ** ---------- .> results:     redis://bioseed_redis:6379/10
- *** --- * --- .> concurrency: 2 (prefork)
-- ******* ---- .> task events: OFF (enable -E to monitor tasks in this worker)
--- ***** ----- 
 -------------- [queues]
                .> fastp            exchange=fastp(direct) key=fastp

In [ ]:
# Where should this code go?

from pathlib import Path

def get_fastp_params(library, seqsample):

    fwd_in_path = seqsample.received[0]
    fwd_in_file = os.path.basename(fwd_in_path)
    fwd_out_file = fwd_in_file.replace('.fastq.gz', '_trimmed.fastq.gz')
    fwd_out_path = os.path.join(library.path, 'trimmed', fwd_out_file)
    rvs_in_path = seqsample.received[1]
    rvs_in_file = os.path.basename(rvs_in_path)
    rvs_out_file = rvs_in_file.replace('.fastq.gz', '_trimmed.fastq.gz')
    rvs_out_path = os.path.join(library.path, 'trimmed', rvs_out_file)
    
    # Normalize Path → str - Celery tasks only accept certain input data types.
    def norm(x): return str(x) if isinstance(x, Path) else x
    
    fastp_params = {
        'path_to_fwd': norm(fwd_in_path),
        'path_to_rev': norm(rvs_in_path),
        'path_to_fwd_out': norm(fwd_out_path),
        'path_to_rev_out': norm(rvs_out_path),
        'threads': 16,
        'polyG':5
    }

    return fastp_params

In [ ]:
from celery import Celery
import os

# Create Celery client
client = Celery(
    'client',
    broker=os.getenv('CELERY_BROKER_URL', 'redis://bioseed_redis:6379/10'),
    backend=os.getenv('CELERY_RESULT_BACKEND', 'redis://bioseed_redis:6379/10')
)

# Submit tasks:

results = []

for sample in seqsamples:
    result = client.send_task(
        'fastp.run',
        kwargs=get_fastp_params(short, sample),
        queue='fastp'
    )
    results.append(result)

In [ ]:
# for i in results:
#     print(i.status)

In [ ]:
all([(r.status=='SUCCESS') for r in results)])

In [ ]:
from aisynbiopipeline.workflows.read_qc import run_multiqc
import shutil

multiqc_report = run_multiqc(short.path / 'trimmed')
multiqc_report_dir = os.path.dirname(multiqc_report)
multiqc_report_file = os.path.basename(multiqc_report)
dst_multiqc_report_file = os.path.join(home_dir, 'trimmed_' + multiqc_report_file)
shutil.copy(multiqc_report, dst_multiqc_report_file)

## 4. Short reads: Breseq (refseq: ACN2586) - Population mode

- Run breseq in polymorphism/population mode on all illumina seqsamples.

In [ ]:
# Three breseq workers are running:

(aisynbio_env) nspahr@seed:~/code/AISynbioPipeline$ python -m aisynbiopipeline.tasks.breseq_task 1
(aisynbio_env) nspahr@seed:~/code/AISynbioPipeline$ python -m aisynbiopipeline.tasks.breseq_task 2
(aisynbio_env) nspahr@seed:~/code/AISynbioPipeline$ python -m aisynbiopipeline.tasks.breseq_task 3

In [ ]:
# Create breseq dir

short.create_subfolder('breseq')

In [ ]:
# List available reference genomes

from aisynbiopipeline.workflows.breseq import list_reference_genomes

list_reference_genomes()

In [ ]:
# Where should this code go?

# Specifies and assigns breseq parameters

def define_breseq_params_pop(seqsample):

    # Normalize Path → str - Celery tasks only accept certain input data types.
    def norm(x): return str(x) if isinstance(x, Path) else x
    
    breseq_params = {
        'read_paths': [norm(x) for x in seqsample.trimmed],
        'breseq_folder': norm(seqsample.breseq),
        'reference': 'ACN2586_NSS.gbk',
        'polymorphism_prediction': True,
        'limit_fold_coverage': 300,
        'num_processors': 4
    }
    return breseq_params

In [ ]:
# Submit tasks

results = []
for sample in shortSeqSamples:
    result = client.send_task(
        'breseq.run',
        kwargs=define_breseq_params_pop(sample),
        queue='breseq'
    )
    results.append(result)

In [ ]:
# for i in results:
#     print(i.status)
#     # print(i.result['output'])

In [ ]:
all([(r.status=='SUCCESS') for r in results)])

## 5. Breseq analysis: Population mode

- Create breseq objects for each seqsample and aggregate summary counts.
- Generate mutation table for all samples and write to csv for import into Google Sheets.
-  [...]

Summary and report files for review will have the following folder structure. Later, check size of folder, possibly zip before downloading to laptop, then uploading to LIMS

In [ ]:
# home_dir/
# ├── read_QC_report.html
# ├── breseq_analysis_populations/
# │   ├── run_summary.csv
# │   ├── all_muations.html
# │   ├── all_mutations.csv (can delete this one after download)
# │   ├── all_mutations_reformatted.csv
# │   ├── summary_all_samples.txt (conmut in all, conmut in some)
# │   └── example_sample_X/
# │       ├── all_mutations_example_sample_X.csv
# │       ├── summary_example_sample_X.txt (conmut in all, conmut in some, increasing pol)
# │       └── polymorphisms_increasing_over_transfers.pdf
# └── breseq_analysis_colonies (IN PLANNING)

In [55]:
# Where should this go?

def parse_seqsample_name(seqsample_name):
    
    import re

    pattern = re.compile(
        r'(TFMN1\.(sohB.pgi|fba.pgi|fba.sohB|sohB.tpiA|fba.tpiA|pgi.tpiA|fba|tpiA|sohB|noDNA|pgi)\.([1-5]))\.T(\d{1,2})\.([PSL])([123])?$'
    )

    match = re.match(pattern, seqsample_name)

    if match:
        sample = str(match.group(1))
        construct = str(match.group(2))
        replicate = int(match.group(3))
        transfer = int(match.group(4))
        isolate = str(match.group(5))
        colony_num = int(match.group(6)) if isinstance(match.group(6), str) else None

    else:
        sample = 'NA'
        construct = 'NA'
        replicate = 'NA'
        transfer = 'NA'
        isolate = 'NA'
        colony_num = 'NA'

    return {'sample': sample, 'construct': construct, 'replicate': replicate, 'transfer': transfer, 'isolate': isolate, 'colony_num': colony_num}
        

In [ ]:
# Create breseq objects for each seqsample and aggregate summary counts

breseq_objects = []

for i in results:
    breseq_folder = i.result['output']
    b = Breseq.from_existing(breseq_folder)
    breseq_objects.append(b)

rows = []

for b in breseq_objects:
    
    row = {}
    
    try:
        b.count_reads()
        b.count_mutations()
    except Exception as e:
        row.update({'seqsample': getattr(b, 'title', None)})
        row.update(parse_seqsample_name(getattr(b, 'title', None)))
        row.update({'error': str(e),
                    'input_read_count': None,
                    'used_read_count': None,
                    'mapped_read_count': None,
                    'consensus_mutation_count': None,
                    'polymorphism_mutation_count': None,}
                    )
        rows.append(row)
        print(f"Error loading Breseq from {b.title}: {e}")
        continue

    row['seqsample'] = getattr(b, 'title', None)
    row.update(parse_seqsample_name(getattr(b, 'title', None)))
    row['error'] = None
    row['input_read_count'] = getattr(b, 'input_read_count', None)
    row['used_read_count'] = getattr(b, 'used_read_count', None)
    row['mapped_read_count'] = getattr(b, 'mapped_read_count', None)
    row['consensus_mutation_count'] = getattr(b, 'consensus_mutation_count', None)
    row['polymorphism_mutation_count'] = getattr(b, 'polymorphism_mutation_count', None)

    rows.append(row)

breseq_summary = pd.DataFrame(rows)
breseq_summary

In [ ]:
# Only expecting one breseq version at this point (namely, the one that was run above).

if len(set([os.path.basename(b.output_folder) for b in breseq_objects])) == 1:
    version_name = list(set([os.path.basename(b.output_folder) for b in breseq_objects]))[0]
else:
    version_name = None
    print("Warning: Not all breseq_objects were run with identical parameters.")

In [ ]:
# Write breseq run summary to csv
breseq_summary.to_csv(os.path.join(breseq_pop_dir, f'breseq_summary_{item_code}_{version_name}.csv'))

In [ ]:
# Create mutation comparison files

from aisynbiopipeline.workflows.breseq import compare_gdiff

reference = 'ACN2586_NSS.gbk'
gdiffs = [b.gd_file for b in breseq_objects]

table_format = 'html'
outfile = os.path.join(breseq_pop_dir, f'all_mutations_{item_code}.{table_format}')
html = compare_gdiff(reference, outfile, gdiffs, format=table_format)

table_format = 'csv'
outfile = os.path.join(breseq_pop_dir, f'all_mutations_{item_code}.{table_format}')
csv = compare_gdiff(reference, outfile, gdiffs, format=table_format)

In [ ]:
compare_df = pd.read_csv(csv)

All of the following transformations should possibly go into a different script...

In [ ]:
# Set display options and table formatting
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

cols_with_diff_vals_for_same_mutation = ['new_read_count', 'new_read_count_basis', 'ref_read_count',
     'ref_read_count_basis', 'multiple_polymorphic_SNPs_in_same_codon',
     'repeat_new_copies', 'repeat_ref_copies'
    ]
cols_uninformative = ['clone', 'mutator_status', 'population', 'time', 'treatment']
cols_to_keep = [col for col in compare_df.columns.to_list() if (col not in cols_with_diff_vals_for_same_mutation) and (col not in cols_uninformative)]

compare_df = compare_df[cols_to_keep]

In [ ]:
# compare_df['experiment'] = compare_df['title'].apply(lambda x: x.split('.')[0])
# compare_df['strain'] = compare_df['title'].apply(lambda x: x.split('.')[1])
# compare_df['colony'] = compare_df['title'].apply(lambda x: x.split('.')[2])

In [ ]:
compare_df.head()

Next, reshape (pivot) the table by grouping by these columns, so that only title (sample) remain as columns (values are the mutation frequencies).
Reindex.
Now ready to analyze

In [ ]:
index = [col for col in compare_df.columns if (col!='title') and (col!='frequency')]

df = compare_df.pivot(index=index, columns = 'title', values = 'frequency').sort_index(level='position')
df = df.fillna(0)
df.head()

In [ ]:
# Write reformatted mutations to csv.
# Decided against formatting like the html table. Will show to group and let them decide which of the cols are important

df.to_csv(os.path.join(breseq_pop_dir, f'mutation_comparison_{item_code}_{version_name}_reformatted.csv'))

In [ ]:
# Consensus mutations (100%) observed in some (any) of the samples

basic_cols = ['position', 'locus_tag', 'gene_name', 'gene_product', 'mutation_category']
consensus_mutations_in_some = df.loc[(df == 1).any(axis=1)].reset_index()[basic_cols + seqsamples]
# consensus_mutations_in_some.to_csv(os.path.join(breseq_pop_dir, f'consensus_mutations_in_some_{item_code}_{version_name}.csv'))

In [ ]:
# Consensus mutations (100%) observed in all of the samples

consensus_mutations_in_all = df.loc[(df == 1).all(axis=1)].reset_index()[basic_cols + seqsamples]
# consensus_mutations_in_all.to_csv(os.path.join(breseq_pop_dir, f'consensus_mutations_in_all_{item_code}_{version_name}.csv'))

In [ ]:
# Write consensus mutations to txt

file = os.path.join(breseq_pop_dir, 'summary_all_samples.txt')

with open(file, "w") as f:
    
    f.write(f"Consensus mutations (100%) observed in SOME (any) of the samples\n")
    for index, row in consensus_mutations_in_some.iterrows():
        f.write(f"{row['locus_tag']}\t{row['gene_name']}\t{row['gene_product']}\t{row['mutation_category']}")

    f.write(f"Consensus mutations (100%) observed in ALL of the samples\n")
    for index, row in consensus_mutations_in_all.iterrows():
        f.write(f"{row['locus_tag']}\t{row['gene_name']}\t{row['gene_product']}\t{row['mutation_category']}")

In [65]:
# Create dataframe of parsed metadata

samples = seqsamples
metadata = pd.DataFrame(pd.Series(seqsamples, name='seqsample'))
metadata['sample'] = metadata['seqsample'].apply(lambda x: parse_seqsample_name(x)['sample'])
metadata['construct'] = metadata['seqsample'].apply(lambda x: parse_seqsample_name(x)['construct'])
metadata['replicate'] = metadata['seqsample'].apply(lambda x: parse_seqsample_name(x)['replicate'])
metadata['transfer'] = metadata['seqsample'].apply(lambda x: parse_seqsample_name(x)['transfer'])
metadata['isolate'] = metadata['seqsample'].apply(lambda x: parse_seqsample_name(x)['isolate'])
metadata['colony_num'] = metadata['seqsample'].apply(lambda x: parse_seqsample_name(x)['colony_num'])
metadata.head()
# sample_order = list(range(3))
# sample_set = dict(zip(sample_order, samples))
# sample_set

,seqsample,sample,construct,replicate,transfer,isolate,colony_num
0,TFMN1.fba.1.T1.P,TFMN1.fba.1,fba,1,1,P,NaN
1,TFMN1.sohB.1.T1.P,TFMN1.sohB.1,sohB,1,1,P,NaN
2,TFMN1.pgi.1.T1.P,TFMN1.pgi.1,pgi,1,1,P,NaN
3,TFMN1.fba.tpiA.1.T1.P,TFMN1.fba.tpiA.1,fba.tpiA,1,1,P,NaN
4,TFMN1.pgi.tpiA.1.T1.P,TFMN1.pgi.tpiA.1,pgi.tpiA,1,1,P,NaN


In [40]:
def sort_key_psl(isolate):
    key = {'P':1, 'S':2, 'L':3}
    return key[isolate]

In [46]:
# Set display options and table formatting
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

In [66]:
metadata['temp_sort_key_psl'] = [sort_key_psl(iso) for iso in metadata['isolate']]
metadata.sort_values(['construct', 'replicate', 'transfer', 'temp_sort_key_psl', 'colony_num'], inplace=True)
metadata.drop(columns='temp_sort_key_psl', inplace=True)
metadata.reset_index(drop=True, inplace=True)
metadata.head(15)

,seqsample,sample,construct,replicate,transfer,isolate,colony_num
0,TFMN1.fba.1.T1.P,TFMN1.fba.1,fba,1,1,P,NaN
1,TFMN1.fba.1.T5.P,TFMN1.fba.1,fba,1,5,P,NaN
2,TFMN1.fba.1.T11.P,TFMN1.fba.1,fba,1,11,P,NaN
3,TFMN1.fba.1.T20.P,TFMN1.fba.1,fba,1,20,P,NaN
4,TFMN1.fba.1.T32.P,TFMN1.fba.1,fba,1,32,P,NaN
5,TFMN1.fba.2.T1.P,TFMN1.fba.2,fba,2,1,P,NaN
6,TFMN1.fba.2.T5.P,TFMN1.fba.2,fba,2,5,P,NaN
7,TFMN1.fba.2.T11.P,TFMN1.fba.2,fba,2,11,P,NaN
8,TFMN1.fba.2.T20.P,TFMN1.fba.2,fba,2,20,P,NaN
9,TFMN1.fba.2.T25.P,TFMN1.fba.2,fba,2,25,P,NaN


In [82]:
metadata.loc[metadata['isolate']!='P']

,seqsample,sample,construct,replicate,transfer,isolate,colony_num
10,TFMN1.fba.2.T25.S1,TFMN1.fba.2,fba,2,25,S,1.0
11,TFMN1.fba.2.T25.S2,TFMN1.fba.2,fba,2,25,S,2.0
12,TFMN1.fba.2.T25.L1,TFMN1.fba.2,fba,2,25,L,1.0
13,TFMN1.fba.2.T25.L2,TFMN1.fba.2,fba,2,25,L,2.0
16,TFMN1.fba.3.T25.S1,TFMN1.fba.3,fba,3,25,S,1.0
17,TFMN1.fba.3.T25.S2,TFMN1.fba.3,fba,3,25,S,2.0
18,TFMN1.fba.3.T25.L1,TFMN1.fba.3,fba,3,25,L,1.0
19,TFMN1.fba.3.T25.L2,TFMN1.fba.3,fba,3,25,L,2.0
31,TFMN1.fba.pgi.2.T8.S1,TFMN1.fba.pgi.2,fba.pgi,2,8,S,1.0
32,TFMN1.fba.pgi.2.T8.S2,TFMN1.fba.pgi.2,fba.pgi,2,8,S,2.0


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

for name, group in metadata.groupby('sample'):

    # For each sample set, make a subfolder in breseq_pop_dir for
    # mutation table, list of interesting mutations, plot of increasing mutations

    sample_dir = os.path.join(breseq_pop_dir, name)
    os.makedir(sample_dir)
    file = os.path.join(sample_dir, f'summary_{name}.txt')
    
    group_seqsamples = group['seqsample'].to_list()
    df_sub = df[group_seqsamples]
    
    # Show fixed mutations present in all seqsamples of this sample
    df_sub_cons_all = df_sub[(df_sub == 1).all(axis=1)].reset_index()
    df_sub_cons_all = df_sub_cons_all[
    [
        'position', 'locus_tag', 'gene_name', 'gene_product', 'mutation_category'
    ] + group_seqsamples]

    # Show fixed mutations present in some seqsamples of this sample
    df_sub_cons_some = df_sub[(df_sub == 1).any(axis=1)].reset_index()
    df_sub_cons_some = df_sub_cons_some[
    [
        'position', 'locus_tag', 'gene_name', 'gene_product', 'mutation_category'
    ] + group_seqsamples]

    # Show polymorphisms observed with increasing frequency over transfers
    df_sub_incr = df_sub[
        # 1. Row-wise non-decreasing
        (df.values[:, 1:] >= df.values[:, :-1]).all(axis=1)
        &
        # 2. First and last cannot be the same
        (df.values[:, 0] != df.values[:, -1])
    ].reset_index()[
    [
        'position', 'locus_tag', 'gene_name', 'gene_product', 'mutation_category'
    ] + seqsamples]

    if len(incr)> df_sub_incr['position'].nunique():
        print(f"In sample {name}, multiple mutations with the same position were found. /nThese may be insertions that are listed as multiple SNPs or there may be multiple alleles at one position.")

    # Print interesting mutations to file
    with open(file, "w") as f:
        
        f.write(f"Consensus mutations present in all seqsamples of {name}\n:")
        for index, row in df_sub_cons_all.iterrows():
            f.write(f"{row['locus_tag']}\t{row['gene_name']}\t{row['gene_product']}\t{row['mutation_category']}")
        f.write('\n')
        f.write(f"Consensus mutations present in some seqsamples of {name}\n:")
        for index, row in df_sub_cons_all.iterrows():
            f.write(f"{row['locus_tag']}\t{row['gene_name']}\t{row['gene_product']}\t{row['mutation_category']}")
        f.write('\n')
        f.write(f"Polymorphisms observed with increasing frequency over transfers of {name}\n:")
        for index, row in df_sub_incr.iterrows():
            f.write(f"{row['locus_tag']}\t{row['gene_name']}\t{row['gene_product']}\t{row['mutation_category']}")

    # Plot polymorphism frequencies that are observed with increasing frequency over transfers
    figures = os.path.join(sample_dir, 'polymorphisms_increasing_over_transfers.pdf')
    
    with PdfPages(figures) as pdf:
        for index, row in incr.iterrows():
            title = f"Position: {row['position']}\nGene name: {row['gene_name']}\nCategory: {row['mutation_category']}" 
            plt.bar(samples, row[samples])
            plt.xticks(rotation=45)
            plt.title(title, loc="left")
            plt.ylabel('frequency')
            plt.show()
            plt.savefig(pdf, format='pdf', bbox_inches='tight')

Show fixed mutations present in all samples of sample set

In [ ]:
# df[(df == 1).all(axis=1)].reset_index()[['position', 'locus_tag', 'gene_name', 'gene_product', 'mutation_category'] + samples]

Show polymorphisms present in all samples of sample set.

In [ ]:
# df[(df!=0).all(axis=1)].reset_index()[['position', 'locus_tag', 'gene_name', 'gene_product', 'mutation_category'] + samples]

Show polymorphisms present in increasing frequency with sample order. (Definitely present this list in the results, because some mutations may appear multiple times and need further investigation.)

In [ ]:
# incr = df[
#     # 1. Row-wise non-decreasing
#     (df.values[:, 1:] >= df.values[:, :-1]).all(axis=1)
#     &
#     # 2. First and last cannot be the same
#     (
#         df.values[:, 0] != df.values[:, -1]
#     )
# ].reset_index()[['position', 'locus_tag', 'gene_name', 'gene_product', 'mutation_category'] + samples]

# if len(incr)> incr['position'].nunique():
#     print("Multiple mutations with the same position were found. /nThese may be insertions that are listed as multiple SNPs or there may be multiple alleles at one position.")

# incr

In [ ]:
# import matplotlib.pyplot as plt

# for index, row in incr.iterrows():
#     title = f"Position: {row['position']}\nGene name: {row['gene_name']}\nCategory: {row['mutation_category']}" 
#     plt.bar(samples, row[samples])
#     plt.xticks(rotation=45)
#     plt.title(title, loc="left")
#     plt.ylabel('frequency')
#     plt.show()

## 4. Short reads: Breseq (refseq: ACN2586) - Consensus mode

- Run breseq in consensus mode on all colony seqsamples.

In [ ]:
# Creating batch (list) of seqsamples for this seqorder

colony_seqsample_names = metadata.loc[metadata['isolate']!= 'P']['seqsample']
seqsamples = [SeqSample(short, name) for name in colony_seqsample_names]

In [ ]:
# Still running?

(aisynbio_env) nspahr@seed:~/code/AISynbioPipeline$ python -m aisynbiopipeline.tasks.breseq_task 1
(aisynbio_env) nspahr@seed:~/code/AISynbioPipeline$ python -m aisynbiopipeline.tasks.breseq_task 2
(aisynbio_env) nspahr@seed:~/code/AISynbioPipeline$ python -m aisynbiopipeline.tasks.breseq_task 3

In [ ]:
# # Create breseq dir

# short.create_subfolder('breseq')

In [ ]:
# Where should this code go?

# Specifies and assigns breseq parameters

def define_breseq_params_colony(seqsample):

    # Normalize Path → str - Celery tasks only accept certain input data types.
    def norm(x): return str(x) if isinstance(x, Path) else x
    
    breseq_params = {
        'read_paths': [norm(x) for x in seqsample.trimmed],
        'breseq_folder': norm(seqsample.breseq),
        'reference': 'ACN2586_NSS.gbk',
        'polymorphism_prediction': False,
        'limit_fold_coverage': 300,
        'num_processors': 4
    }
    return breseq_params

In [ ]:
# Submit tasks

results = []
for sample in shortSeqSamples:
    result = client.send_task(
        'breseq.run',
        kwargs=define_breseq_params_colony(sample),
        queue='breseq'
    )
    results.append(result)

In [ ]:
# for i in results:
#     print(i.status)
#     # print(i.result['output'])

In [ ]:
all([(r.status=='SUCCESS') for r in results)])

## 5. Breseq analysis: Consensus mode

- Create breseq objects for each seqsample and aggregate summary counts.
- Generate mutation table for all samples and write to csv for import into Google Sheets.
-  [...]

- Retrieve all colony samples and run breseq in consensus mode.
- Create objects of these breseq results and compare all v all, write to file.
- Group colony samples by transfer sample, and list all mutations that they have in common.

Summary and report files for review will have the following folder structure. Later, check size of folder, possibly zip before downloading to laptop, then uploading to LIMS

In [ ]:
# home_dir/
# ├── read_QC_report.html
# ├── breseq_analysis_populations/
# │   ├── run_summary.csv
# │   ├── all_muations.html
# │   ├── all_mutations.csv (can delete this one after download)
# │   ├── all_mutations_reformatted.csv
# │   ├── summary_all_samples.txt (conmut in all, conmut in some)
# │   └── example_sample_X/
# │       ├── all_mutations_example_sample_X.csv
# │       ├── summary_example_sample_X.txt (conmut in all, conmut in some, increasing pol)
# │       └── polymorphisms_increasing_over_transfers.pdf
# └── breseq_analysis_colonies (IN PLANNING)

In [ ]:
# Where should this go?

def parse_seqsample_name(seqsample_name):
    
    import re

    pattern = re.compile(
        r'(TFMN1\.(sohB.pgi|fba.pgi|fba.sohB|sohB.tpiA|fba.tpiA|pgi.tpiA|fba|tpiA|sohB|noDNA|pgi)\.([1-5]))\.T(\d{1,2})\.([PSL])([123])?$'
    )

    match = re.match(pattern, seqsample_name)

    if match:
        sample = str(match.group(1))
        construct = str(match.group(2))
        replicate = int(match.group(3))
        transfer = int(match.group(4))
        isolate = str(match.group(5))
        colony_num = int(match.group(6)) if isinstance(match.group(6), str) else None

    else:
        sample = 'NA'
        construct = 'NA'
        replicate = 'NA'
        transfer = 'NA'
        isolate = 'NA'
        colony_num = 'NA'

    return {'sample': sample, 'construct': construct, 'replicate': replicate, 'transfer': transfer, 'isolate': isolate, 'colony_num': colony_num}
        

In [ ]:
# Create breseq objects for each seqsample and aggregate summary counts

breseq_objects = []

for i in results:
    breseq_folder = i.result['output']
    b = Breseq.from_existing(breseq_folder)
    breseq_objects.append(b)

rows = []

for b in breseq_objects:
    
    row = {}
    
    try:
        b.count_reads()
        b.count_mutations()
    except Exception as e:
        row.update({'seqsample': getattr(b, 'title', None)})
        row.update(parse_seqsample_name(getattr(b, 'title', None)))
        row.update({'error': str(e),
                    'input_read_count': None,
                    'used_read_count': None,
                    'mapped_read_count': None,
                    'consensus_mutation_count': None,
                    'polymorphism_mutation_count': None,}
                    )
        rows.append(row)
        print(f"Error loading Breseq from {b.title}: {e}")
        continue

    row['seqsample'] = getattr(b, 'title', None)
    row.update(parse_seqsample_name(getattr(b, 'title', None)))
    row['error'] = None
    row['input_read_count'] = getattr(b, 'input_read_count', None)
    row['used_read_count'] = getattr(b, 'used_read_count', None)
    row['mapped_read_count'] = getattr(b, 'mapped_read_count', None)
    row['consensus_mutation_count'] = getattr(b, 'consensus_mutation_count', None)
    row['polymorphism_mutation_count'] = getattr(b, 'polymorphism_mutation_count', None)

    rows.append(row)

breseq_summary = pd.DataFrame(rows)
breseq_summary

In [ ]:
# Only one breseq version (namely, the one that was run last).

if len(set([os.path.basename(b.output_folder) for b in breseq_objects])) == 1:
    version_name = list(set([os.path.basename(b.output_folder) for b in breseq_objects]))[0]
else:
    version_name = None
    print("Warning: Not all breseq_objects were run with identical parameters.")

In [ ]:
# Write breseq run summary to csv
breseq_summary.to_csv(os.path.join(breseq_col_dir, f'breseq_summary_{item_code}_{version_name}.csv'))

In [ ]:
# Create mutation comparison files

from aisynbiopipeline.workflows.breseq import compare_gdiff

reference = 'ACN2586_NSS.gbk'
gdiffs = [b.gd_file for b in breseq_objects]

table_format = 'html'
outfile = os.path.join(breseq_pop_dir, f'all_mutations_{item_code}.{table_format}')
html = compare_gdiff(reference, outfile, gdiffs, format=table_format)

table_format = 'csv'
outfile = os.path.join(breseq_pop_dir, f'all_mutations_{item_code}.{table_format}')
csv = compare_gdiff(reference, outfile, gdiffs, format=table_format)

In [ ]:
compare_df = pd.read_csv(csv)

All of the following transformations should possibly go into a different script...

In [ ]:
# Set display options and table formatting
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

cols_with_diff_vals_for_same_mutation = ['new_read_count', 'new_read_count_basis', 'ref_read_count',
     'ref_read_count_basis', 'multiple_polymorphic_SNPs_in_same_codon',
     'repeat_new_copies', 'repeat_ref_copies'
    ]
cols_uninformative = ['clone', 'mutator_status', 'population', 'time', 'treatment']
cols_to_keep = [col for col in compare_df.columns.to_list() if (col not in cols_with_diff_vals_for_same_mutation) and (col not in cols_uninformative)]

compare_df = compare_df[cols_to_keep]

In [ ]:
# compare_df['experiment'] = compare_df['title'].apply(lambda x: x.split('.')[0])
# compare_df['strain'] = compare_df['title'].apply(lambda x: x.split('.')[1])
# compare_df['colony'] = compare_df['title'].apply(lambda x: x.split('.')[2])

In [ ]:
compare_df.head()

Next, reshape (pivot) the table by grouping by these columns, so that only title (sample) remain as columns (values are the mutation frequencies).
Reindex.
Now ready to analyze

In [ ]:
index = [col for col in compare_df.columns if (col!='title') and (col!='frequency')]

df = compare_df.pivot(index=index, columns = 'title', values = 'frequency').sort_index(level='position')
df = df.fillna(0)
df.head()

In [ ]:
# Write reformatted mutations to csv.
# Decided against formatting like the html table. Will show to group and let them decide which of the cols are important

df.to_csv(os.path.join(breseq_col_dir, f'mutation_comparison_{item_code}_{version_name}_reformatted.csv'))

In [ ]:
# Create dataframe of parsed metadata

samples = seqsamples
metadata = pd.DataFrame(pd.Series(seqsamples, name='seqsample'))
metadata['sample'] = metadata['seqsample'].apply(lambda x: parse_seqsample_name(x)['sample'])
metadata['construct'] = metadata['seqsample'].apply(lambda x: parse_seqsample_name(x)['construct'])
metadata['replicate'] = metadata['seqsample'].apply(lambda x: parse_seqsample_name(x)['replicate'])
metadata['transfer'] = metadata['seqsample'].apply(lambda x: parse_seqsample_name(x)['transfer'])
metadata['isolate'] = metadata['seqsample'].apply(lambda x: parse_seqsample_name(x)['isolate'])
metadata['colony_num'] = metadata['seqsample'].apply(lambda x: parse_seqsample_name(x)['colony_num'])
metadata.head()
# sample_order = list(range(3))
# sample_set = dict(zip(sample_order, samples))
# sample_set

In [ ]:
def sort_key_psl(isolate):
    key = {'P':1, 'S':2, 'L':3}
    return key[isolate]

In [ ]:
# Set display options and table formatting
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

In [ ]:
metadata['temp_sort_key_psl'] = [sort_key_psl(iso) for iso in metadata['isolate']]
metadata.sort_values(['construct', 'replicate', 'transfer', 'temp_sort_key_psl', 'colony_num'], inplace=True)
metadata.drop(columns='temp_sort_key_psl', inplace=True)
metadata.reset_index(drop=True, inplace=True)
metadata.head(15)

In [ ]:
metadata = metadata.loc[metadata['isolate']!='P']

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

for name, group in metadata.groupby(['sample', 'transfer', 'isolate']):
    group_seqsamples = group['seqsample'].to_list()
    df_sub = df[group_seqsamples]
    co_occurring_mutations = df_sub.loc[(df_sub == 1).all(axis=1)].reset_index()
    
    print(f'Co-occuring mutations in colony seqsamples {[print(s+', ' for s in group['seqsample']]}:')
    for index, row in co_occurring_mutations.iterrows():
        print(f"{row['locus_tag']}\t{row['gene_name']}\t{row['gene_product']}\t{row['mutation_category']}")
    